# feature engineering

On base of the tree messages, one conversation becones one dataset row with relevant information like numbers of token, etc. in a dataframe.

In [1]:
import json
import re
import random
import numpy as np
from pathlib import Path
from time import perf_counter
import tiktoken
import pandas as pd
from spellchecker import SpellChecker
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

In [2]:
# load dataset

def load_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


processed_path = Path(
    "C:/Users/heike/Desktop/Stackfuel/Portfolio/llm-sustainability-analysis/01_data/02_processed/sharegpt_english.json"
)

english_data = load_json(processed_path)

print(f"Loaded conversations: {len(english_data)}")

Loaded conversations: 53824


In [3]:
# conversation ids

conversation_ids = [
    item.get("id")
    for item in english_data
    if isinstance(item, dict) and item.get("id") is not None
]

print(f"Conversation IDs: {len(conversation_ids)}")
print(f"Unique Conversation IDs: {len(set(conversation_ids))}")
print(f"Duplicates: {len(conversation_ids) - len(set(conversation_ids))}")
    

Conversation IDs: 53824
Unique Conversation IDs: 53824
Duplicates: 0


In [4]:
# build dataframe

rows = []

encoding = tiktoken.get_encoding("cl100k_base")
spell = SpellChecker(language="en")


# =========================
# helper functions
# =========================

def get_first_prompt_response_pair(tree):
    conversations = tree.get("conversations", [])

    for i in range(len(conversations) - 1):
        current = conversations[i]
        next_msg = conversations[i + 1]

        if current.get("from") == "human" and next_msg.get("from") == "gpt":
            return current.get("value", ""), next_msg.get("value", "")

    return "", ""


def get_words(prompt):
    return re.findall(r"[A-Za-z']+", prompt.lower())


# =========================
# token-based features
# =========================

def count_tokens(text):
    return len(
        encoding.encode(
            text,
            disallowed_special=()
        )
    )


def count_user_prompts(tree):
    return sum(
        message.get("from") == "human"
        for message in tree.get("conversations", [])
    )


def count_total_user_tokens(tree):
    total = 0

    for message in tree.get("conversations", []):
        if message.get("from") == "human":
            total += count_tokens(message.get("value", ""))

    return total


def count_total_assistant_tokens(tree):
    total = 0

    for message in tree.get("conversations", []):
        if message.get("from") == "gpt":
            total += count_tokens(message.get("value", ""))

    return total


# =========================
# prompt design features
# =========================

def has_role_instruction(prompt):
    prompt = prompt.lower()

    role_patterns = [
        "act as",
        "you are",
        "you're",
        "pretend you are",
        "imagine you are",
        "take the role",
        "role of",
    ]

    return any(pattern in prompt for pattern in role_patterns)


def has_audience_or_level_instruction(prompt):
    prompt = prompt.lower()

    patterns = [
        "explain it to me like",
        "explain like i am",
        "explain like i'm",
        "eli5",
        "like i am 5",
        "like i'm 5",
        "like i am 10",
        "like i'm 10",
        "for beginners",
        "to a beginner",
        "in simple terms",
        "plain english",
        "non-technical",
        "for a child",
        "for a high school student",
    ]

    return any(pattern in prompt for pattern in patterns)


def has_format_instruction(prompt):
    prompt = prompt.lower()

    format_keywords = [
        "bullet point",
        "bullet points",
        "table",
        "json",
        "csv",
        "list",
        "markdown",
        "format",
        "section",
        "sections",
        "outline",
        "step by step",
        "numbered",
    ]

    return any(keyword in prompt for keyword in format_keywords)


def count_questions(prompt):
    return prompt.count("?")


def prompt_style(prompt):
    prompt_lower = prompt.lower().strip()

    instruction_verbs = [
        "write", "summarize", "explain", "create", "generate", "translate",
        "list", "compare", "analyze", "give", "make", "draft", "compose",
        "rewrite", "classify", "extract", "find", "calculate",
    ]

    if "?" in prompt_lower:
        return "question"

    if any(prompt_lower.startswith(verb) for verb in instruction_verbs):
        return "instruction"

    return "other"


# =========================
# orthographic features
# =========================

def orthographic_error_rate(prompt):
    words = get_words(prompt)

    if not words:
        return 0

    unknown_words = spell.unknown(words)

    return len(unknown_words) / len(words)


# =========================
# task type
# =========================

def classify_task_type(prompt):
    prompt = prompt.lower()

    if any(word in prompt for word in ["python", "java", "javascript", "code", "function", "bug", "error", "api", "sql", "html", "css"]):
        return "coding"

    if any(word in prompt for word in ["email", "subject line", "reply to", "newsletter"]):
        return "email_writing"

    if any(word in prompt for word in ["summarize", "summary", "tl;dr", "key points"]):
        return "summarization"

    if any(word in prompt for word in ["translate", "translation", "in english", "into english", "into spanish", "into french"]):
        return "translation"

    if any(word in prompt for word in ["explain", "what is", "how does", "why does", "difference between", "simple terms"]):
        return "explanation"

    if any(word in prompt for word in ["write", "draft", "compose", "story", "poem", "article", "script", "blog post"]):
        return "writing_generation"

    if any(word in prompt for word in ["idea", "ideas", "brainstorm", "suggest", "recommend"]):
        return "brainstorming"

    if any(word in prompt for word in ["act as", "pretend", "roleplay", "you are now", "simulate"]):
        return "roleplay"

    return "general_assistance"


# =========================
# build conversation-level rows
# =========================

for tree in english_data:

    first_prompt, first_response = get_first_prompt_response_pair(tree)

    total_turns = len(tree.get("conversations", []))
    user_prompts = count_user_prompts(tree)

    total_user_tokens = count_total_user_tokens(tree)
    total_assistant_tokens = count_total_assistant_tokens(tree)
    total_tokens = total_user_tokens + total_assistant_tokens
    log_total_tokens = np.log1p(total_tokens)

    follow_up_prompts = max(0, user_prompts - 1)
    needs_follow_up = follow_up_prompts > 0

    rows.append({
        "conversation_id": tree.get("id"),

        # CORE PAIR
        "first_prompt": first_prompt,
        "first_response": first_response,

        # TOKENS
        "first_prompt_tokens": count_tokens(first_prompt),
        "first_response_tokens": count_tokens(first_response),

        # STRUCTURE
        "total_turns": total_turns,
        "interaction_rounds": total_turns / 2,

        # COST SIGNALS
        "total_user_tokens": total_user_tokens,
        "total_assistant_tokens": total_assistant_tokens,
        "total_tokens": total_tokens,
        "log_total_tokens": log_total_tokens,

        # BEHAVIOR
        "follow_up_prompts": follow_up_prompts,
        "needs_follow_up": needs_follow_up,

        # PROMPT FEATURES
        "has_role_instruction": has_role_instruction(first_prompt),
        "has_audience_or_level_instruction": has_audience_or_level_instruction(first_prompt),
        "has_format_instruction": has_format_instruction(first_prompt),
        "question_count": count_questions(first_prompt),
        "prompt_style": prompt_style(first_prompt),

        # TASK
        "task_type": classify_task_type(first_prompt),

        # NOISE
        "orthographic_error_rate": orthographic_error_rate(first_prompt),
    })


df = pd.DataFrame(rows)

boolean_columns = [
    "has_role_instruction",
    "has_audience_or_level_instruction",
    "has_format_instruction",
]

df[boolean_columns] = df[boolean_columns].astype(int)
df["needs_follow_up"] = df["needs_follow_up"].astype(int)
df["question_count"] = df["question_count"].astype(int)

df.head()

# sanity check
len(df), len(english_data)

(53824, 53824)

In [5]:
print("task_type", df["task_type"].value_counts())

task_type task_type
general_assistance    22372
coding                11684
writing_generation     9040
explanation            4721
brainstorming          1933
email_writing          1221
summarization          1074
roleplay                958
translation             821
Name: count, dtype: int64


In [6]:
print("has_role_instruction", df["has_role_instruction"].value_counts())
print("has_audience_or_level_instruction", df["has_audience_or_level_instruction"].value_counts())
print("has_format_instruction", df["has_format_instruction"].value_counts())
print("question_count", df["question_count"].describe())
print("prompt_style", df["prompt_style"].value_counts())
print("orthographic_error_rate", df["orthographic_error_rate"].describe())
print("task_type", df["task_type"].value_counts())

has_role_instruction has_role_instruction
0    47269
1     6555
Name: count, dtype: int64
has_audience_or_level_instruction has_audience_or_level_instruction
0    53372
1      452
Name: count, dtype: int64
has_format_instruction has_format_instruction
0    41441
1    12383
Name: count, dtype: int64
question_count count    53824.000000
mean         0.559267
std          2.725067
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max        218.000000
Name: question_count, dtype: float64
prompt_style prompt_style
other          26427
question       18457
instruction     8940
Name: count, dtype: int64
orthographic_error_rate count    53824.000000
mean         0.052800
std          0.076313
min          0.000000
25%          0.000000
50%          0.022173
75%          0.076923
max          1.000000
Name: orthographic_error_rate, dtype: float64
task_type task_type
general_assistance    22372
coding                11684
writing_generation     9040
explana

In [7]:
print(df.shape)
#df.head(5)

(53824, 20)


## topic modelling

This step tries to find out the topics in the prompts. NMF will be executed, labels will be added and random checks will be executed.

In [8]:
texts = df["first_prompt"].fillna("")     # only first prompt, otherwise data leakage in models

In [9]:
# initiate TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    min_df=10,
    max_df=0.8,
    ngram_range=(1, 2)
)

In [10]:
# fit and transform vectorizer

tfidf = vectorizer.fit_transform(texts)

In [11]:
# fit and transform model

nmf = NMF(n_components=8, random_state=42)
topic_matrix = nmf.fit_transform(tfidf)

In [12]:
df["topic_id"] = topic_matrix.argmax(axis=1)

In [13]:
feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(nmf.components_):
    top_indices = topic.argsort()[-10:][::-1]
    top_words = [feature_names[i] for i in top_indices]

    print(f"Topic {topic_idx}: {', '.join(top_words)}")

Topic 0: want, use, like, help, need, create, questions, provide, business, ai
Topic 1: dan, chatgpt, character, responses, dan responses, answer, pretend, information, date time, respond
Topic 2: write, story, style, poem, english, script, language, article, english language, write english
Topic 3: explain, simple, simple terms, quantum, quantum computing, computing simple, explain quantum, terms, computing, examples
Topic 4: code, python, using, data, file, create, api, python code, function, write python
Topic 5: know, game, like, don, hi, let, don know, just, book, let know
Topic 6: https, com, www, url, https www, results, search, url https, search results, using
Topic 7: make, javascript, http request, http, request, game, tell, story, like, make sure


In [14]:
# manual labelling

topic_labels = {
    0: "general_business_assistance",
    1: "jailbreak_roleplay",
    2: "creative_writing",
    3: "explanation",
    4: "coding",
    5: "casual_chat_or_games",
    6: "web_search_or_url_task",
    7: "prompting_or_questions",
}


df["topic_label"] = df["topic_id"].map(topic_labels)

In [15]:
# random checks

pd.set_option("display.max_colwidth", None)

examples = (
    df.groupby("topic_id", group_keys=False)
    .sample(n=3, random_state=42)
)

#examples[["topic_id", "conversation_id", "first_prompt"]]


## targets "quality" and "efficiency"

In [16]:
### old quality - do not use

def has_code(text):
    return 1 if "```" in text else 0


def has_bullets(text):
    return 1 if re.search(r"(\n- |\n\* )", text) else 0


def has_steps(text):
    return 1 if re.search(r"\b(step|first|second|finally|1\.|2\.)\b", text.lower()) else 0


def repetition_ratio(text):
    words = text.lower().split()
    if len(words) == 0:
        return 0
    unique_words = len(set(words))
    return 1 - (unique_words / len(words))


def normalize(x):
    return np.log1p(x)

In [17]:
def extract_quality_features(prompt, response):
    features = {}

    # Structure
    features["structure"] = (
        has_code(response)
        + has_bullets(response)
        + has_steps(response)
    ) / 3

    # Informativeness (proxy)
    features["informativeness"] = normalize(len(response.split())) / (len(prompt.split()) + 1)

    # Clarity (inverse repetition)
    features["clarity"] = 1 - repetition_ratio(response)

    # Conciseness (penalty for too long answers)
    features["conciseness"] = 1 / normalize(len(response.split()) + 1)

    # Task fit (simple heuristic proxy)
    features["task_fit"] = 1 if len(response) > len(prompt) else 0

    return features

In [18]:
def compute_quality(prompt, response):
    f = extract_quality_features(prompt, response)

    quality = (
        0.30 * f["structure"] +
        0.25 * f["informativeness"] +
        0.20 * f["clarity"] +
        0.15 * f["conciseness"] +
        0.10 * f["task_fit"]
    )

    return quality

In [19]:
def add_quality_column(df):
    df["quality"] = df.apply(
        lambda row: compute_quality(row["prompt"], row["response"]),
        axis=1
    )
    return df


In [20]:
df["quality"] = df.apply(
    lambda r: compute_quality(r["first_prompt"], r["first_response"]),
    axis=1
)

df["efficiency"] = df["quality"] / np.log1p(df["total_tokens"])

df[["efficiency", "quality"]].describe()


df["legacy_quality"] = df["quality"]
df["legacy_efficiency"] = df["efficiency"]

del df["quality"]
del df["efficiency"]



In [21]:
df["target_cost"] = np.log1p(df["total_tokens"])
df["target_depth"] = np.log1p(df["total_turns"])
df["target_success"] = 1 - df["needs_follow_up"].astype(int)
#df.head()

In [22]:
base_path = Path(
    "C:/Users/heike/Desktop/Stackfuel/Portfolio/llm-sustainability-analysis/01_data/03_features/conversation_features.csv"
)

base_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(base_path, index=False)